In [11]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready!')

Libraries ready!


In [12]:
from groq import Groq
API_KEY = "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"
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"
print(f'Groq client configured with model: {MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [13]:
def ask_llm(user_message, system_message="You are a helpful assistant",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content
test_response = ask_llm(
    "What is ELT in data engineering? Answer in exactly 2 sentences."
)
test_response2 = ask_llm(
    "Is GenAI and Data Engineering a good career in 2026?"
)
print("=== LLM Response ===")
print(test_response)
print("\n=== LLM Response 2 ===")
print(test_response2)

=== LLM Response ===
ELT stands for Extract, Load, Transform, which is a data engineering pipeline that combines the best practices of ETL (Extract, Transform, Load) and adds an extra step of loading data directly into its final form, often into a data warehouse or target system. ELT is commonly used in big data and cloud-based architectures, where data is processed in its raw form and then loaded into a target system, making it easier to manage and analyze large datasets.

=== LLM Response 2 ===
As of my cut-off knowledge in 2023, GenAI (Generative Artificial Intelligence) and Data Engineering are indeed promising and in-demand fields in the tech industry. Here's why:

**GenAI:**

1. **Growing demand**: GenAI has numerous applications in areas like content generation, chatbots, predictive analytics, and more. As AI continues to evolve, the demand for GenAI experts will likely increase.
2. **Advancements in NLP**: Recent breakthroughs in Natural Language Processing (NLP) have made GenA

In [14]:
response_elt = ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture"
    "(Bronze,Silver,Gold_layers) realtes to ELT pipeline.",
    system_message="You are a senior data engineering Instructor."
                   "Be concise and practical."
)
print('Medallion + ELT connection:')
print(response_elt)
print()
print('--- Token explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_elt.split())*1.3,'tokens.')
print('Llma-3.1-8b context window:8192 tokens(-6000 words per conversation)')

Medallion + ELT connection:
As a senior data engineering instructor, I'll explain the Medallion Architecture (Bronze, Silver, Gold layers) and its relation to the ELT (Extract, Load, Transform) pipeline in 3 bullet points:

• **Bronze Layer (Raw Data):** This layer corresponds to the **Extract** step in the ELT pipeline. It's where raw data from various sources is ingested and stored in its native format, often in a NoSQL database, data lake, or object store. The primary focus is on data ingestion, not transformation or processing.

• **Silver Layer (Processed Data):** This layer is equivalent to the **Load** step in the ELT pipeline. It involves processing the raw data from the Bronze layer, performing data quality checks, and aggregating data into a more structured format. The output is stored in a relational database, data warehouse, or data mart.

• **Gold Layer (Curated Data):** This layer aligns with the **Transform** step in the ELT pipeline. It's where the processed data from t

In [15]:
zero_shot_response = ask_llm(
    "Extract the city name from this address: "
    "456 Bridgen Road, Bangalore 560025, Karnataka, India"
)

print("Zero-Shot Result:")
print(zero_shot_response)
print()

ambiguous_response = ask_llm(
    "Clean this data: ramesh kumar, 4500, mumbai"
)

print("Ambiguous Zero-Shot Result:")
print(ambiguous_response)
print()

print("Problem: Output format is unpredictable and not machine-parseable!")

Zero-Shot Result:
The city name extracted from the address is: Bangalore

Ambiguous Zero-Shot Result:
To clean this data, I will make an assumption that it is a person's name, age, and city, and I will format it for better readability. Here's the cleaned data:

**Name:** Ramesh Kumar
**Age:** 45 (assuming the number 4500 is not the actual age, could be an ID, salary, or some other metric. Age is often in the range of 0-120)
**City:** Mumbai

If you intended the number 4500 to represent an actual age, it's unlikely for a person to be 4500 years old. However, if it represents something else, please provide context or clarify what 4500 represents.

Problem: Output format is unpredictable and not machine-parseable!


In [16]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input: RAMESH KUMAR,45000,mumbai
output:{"name":"Ramesh","salary":45000,"city":"Mumbai"}
Input:priya nair, 52000,Delhi
output:{"name":"Priya Nair","salary":52000,"city":"Delhi"}
Now convert this:
Input:ANANYA DAS, 30000,kolkata
Output:"""

few_shot_response=ask_llm(
    few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()
try:
  parsed = json.loads(few_shot_response.strip())
  print('Successfully parsed as JSON!')
  print(f'Name:{parsed["name"]},salary:{parsed['salary']},city:{parsed['city']}')
except json.JSONDecodeError:
        print('Parsing failed - model added extra text')
        print('Solution: add explicit instructions in the prompt')

Few-Shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(',')

    # Create a dictionary with the given keys
    employee = {
        "name": values[0].strip().title(),
        "salary": int(values[1].strip()),
        "city": values[2].strip().title()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
employee_text = "ANANYA DAS, 30000, kolkata"
print(convert_to_json(employee_text))
```

When you run this code, it will output:

```json
{
    "name": "Ananya Das",
    "salary": 30000,
    "city": "Kolkata"
}
```

This code works by splitting the input string into individual values using the comma as a delimiter. It then creates a dictionary with the given keys and assigns the corresponding values. Finally, it converts the 

In [17]:
same_question ="Review this Python code and identify any issues:\n"
"df['revenue]=df['qty]*df['price]\n"
"result= df.groupby('dept').sum()"

generic_response = ask_llm(same_question, temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()

role_response = ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of prediction"
    "experience. Review code critically for production readiness"
    "data type issues, and potential failures at scale",
    temperature=0.2
)
print('with Role Prompting (Senior Data Engineer)')
print(role_response[:400],'...')
print()
print('Notice : role prompting produces more technical, actionable feedback')

Without Role Prompting:
I'm ready to help. Please go ahead and provide the Python code you'd like me to review. I'll do my best to identify any issues, suggest improvements, and provide explanations for my findings. ...

with Role Prompting (Senior Data Engineer)
However, you haven't provided any Python code yet. Please paste the code you'd like me to review, and I'll do my best to identify any potential issues related to data type, scalability, and production readiness.

Once you provide the code, I'll review it critically and provide feedback on:

1. Data type issues
2. Potential failures at scale
3. Suggestions for improvement

Please paste the code, an ...

Notice : role prompting produces more technical, actionable feedback


In [18]:
prompt ="Give me one creative name for a data analytics startup"
print('=== Temperature Experiment ===')
for temp in [0.0,0.5,1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f'Temperature={temp}:{response.strip()}')
  time.sleep(1)
print()
print('Observation:')
print('  temperature=0.0 -> same or very similar answer every run(deterministic)')
print('  temperature=0.5 -> some variation')
print('  temperature=1.0 more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature =0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE ouput - not creative variation')


=== Temperature Experiment ===
Temperature=0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.

Alternatively, if you'd like more options, I can provide you with a list of creative names for a data analytics startup.
Temperature=0.5:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the startup's ability to connect data points and provide valuable insights. It also has a modern and sleek sound to it.
Temperature=1.0:Here's a creative name for a data analytics startup:

**NexaVisa**

Breakdown:
- "Nexa" comes from the word "nexus," representing connections and networks. 
- "Visa" implies vision, a nod to the idea of seeing and understa

In [19]:
import json

invoice_text = (
    "Invoice #2024-001 from TECHWORLD SOLUTIONS "
    "dated 15th January 2024, Amount: Rs. 45000 for Laptop"
)

weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)

print("WEAK PROMPT OUTPUT:")
print(weak_response)
print()

try:
    json.loads(weak_response)
    print("PARSEABLE: Yes")
except:
    print('PARSEABLE: No - Cannot load into DataFrame')
print('\n' + '='*50 + '\n')
strong_system ="""You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object,
Do NOT include any explanation, premble, or markdown formatting.
Return ONLY the JSON, nothing else."""

WEAK PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Information:**

- **Invoice Number:** 2024-001
- **Invoice Date:** 15th January 2024
- **Invoice Provider:** TECHWORLD SOLUTIONS
- **Invoice Description:** Laptop
- **Invoice Amount:** Rs. 45,000

Let me know if you need any further assistance.

PARSEABLE: No - Cannot load into DataFrame




In [20]:
import json

invoice_text = (
    "Invoice #2024-001 from TECHWORLD SOLUTIONS "
    "dated 15th January 2024, Amount: Rs. 45000 for Laptop"
)

# Weak Prompt
weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)

print("WEAK PROMPT OUTPUT:")
print(weak_response)
print()

try:
    json.loads(weak_response)
    print("PARSEABLE: Yes")
except json.JSONDecodeError:
    print("PARSEABLE: No - Cannot load into DataFrame")

print("\n" + "=" * 50 + "\n")

# Strong Prompt
strong_system = """
You are a data extraction specialist for an accounting pipeline.

Extract invoice data and return ONLY a valid JSON object.
Do NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing else.

JSON schema:
{
  "invoice_id": "",
  "vendor_name": "",
  "amount": 0,
  "currency": "INR",
  "invoice_date": "",
  "category": ""
}
"""

strong_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    system_message=strong_system,
    temperature=0
)

print("STRONG PROMPT OUTPUT:")
print(strong_response)
print()

try:
    parsed = json.loads(strong_response.strip())

    print("PARSEABLE: Yes")
    print(f"Invoice ID : {parsed['invoice_id']}")
    print(f"Vendor     : {parsed['vendor_name']}")
    print(f"Amount     : {parsed['amount']}")
    print(f"Currency   : {parsed['currency']}")
    print(f"Date       : {parsed['invoice_date']}")
    print(f"Category   : {parsed['category']}")

except json.JSONDecodeError:
    print("PARSEABLE: No")


WEAK PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** January 15, 2024
- **Vendor:** Techworld Solutions
- **Amount:** Rs. 45,000
- **Description:** Laptop

I have reformatted the date to a standard format and separated the amount into the currency and the numerical value.

PARSEABLE: No - Cannot load into DataFrame


STRONG PROMPT OUTPUT:
{
  "invoice_id": "2024-001",
  "vendor_name": "TECHWORLD SOLUTIONS",
  "amount": 45000,
  "currency": "INR",
  "invoice_date": "2024-01-15",
  "category": "Laptop"
}

PARSEABLE: Yes
Invoice ID : 2024-001
Vendor     : TECHWORLD SOLUTIONS
Amount     : 45000
Currency   : INR
Date       : 2024-01-15
Category   : Laptop
